In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix



[TensorFlow DLL Diagnostic] Analyzing: c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
#  LOAD THE DATASET

print("Loading dataset...")
df = pd.read_csv('adhd_data.csv')

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nTarget column 'Diagnosis_Class' unique values: {df['Diagnosis_Class'].unique()}")


Loading dataset...
Dataset shape: (6500, 14)
Columns: ['Age', 'Gender', 'Educational_Level', 'Family_History', 'Sleep_Hours', 'Daily_Activity_Hours', 'Diagnosis_Class', 'Daily_Phone_Usage_Hours', 'Daily_Walking_Running_Hours', 'Difficulty_Organizing_Tasks', 'Focus_Score_Video', 'Daily_Coffee_Tea_Consumption', 'Learning_Difficulties', 'Anxiety_Depression_Levels']

First 5 rows:
   Age  Gender Educational_Level Family_History  Sleep_Hours  \
0    8       1           Primary             No            8   
1    9       2           Primary             No           11   
2    9       1           Primary             No            9   
3    5       2      Kindergarten            Yes            7   
4   13       1            Middle             No            3   

   Daily_Activity_Hours  Diagnosis_Class  Daily_Phone_Usage_Hours  \
0                     7                0                        2   
1                     7                3                        2   
2                     5     

In [ ]:
#  PREPROCESS THE DATA

print("\n" + "="*50)
print("PREPROCESSING DATA")
print("="*50)


PREPROCESSING DATA


In [ ]:
# Separate features and target
target_column = 'Diagnosis_Class'
X = df.drop(columns=[target_column])
y = df[target_column]

In [ ]:
# Handle categorical columns (convert text to numbers)
categorical_columns = X.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(categorical_columns)}")

Categorical columns: ['Educational_Level', 'Family_History']


In [ ]:
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"  - {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")


  - Educational_Level: {'Kindergarten': 0, 'Middle': 1, 'Not Working': 2, 'Primary': 3, 'Secondary': 4, 'University': 5, 'Working': 6}
  - Family_History: {'No': 0, 'Unknown': 1, 'Yes': 2}


In [ ]:
# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")



Training set size: 5200
Test set size: 1300


In [ ]:
# Normalize/Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled to mean=0, std=1")

Features scaled to mean=0, std=1


In [ ]:
# 3. BUILD TENSORFLOW MODEL

print("\n" + "="*50)
print("BUILDING TENSORFLOW MODEL")
print("="*50)

# Determine number of classes
num_classes = len(y.unique())
print(f"Number of classes: {num_classes}")


BUILDING TENSORFLOW MODEL
Number of classes: 4


In [ ]:
# Convert labels to categorical (one-hot encoding)
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)


In [ ]:
# Build the model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])


In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,260 (47.89 KB)

 Trainable params: 12,260 (47.89 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 4. TRAIN THE MODEL

print("\n" + "="*50)
print("TRAINING MODEL")
print("="*50)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=50,
    batch_size=32,
    validation_split=0.2,  # Use 20% of training data for validation
    verbose=1
)


TRAINING MODEL
Epoch 1/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4593 - loss: 1.2234 - val_accuracy: 0.5894 - val_loss: 0.8789
Epoch 2/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5958 - loss: 0.8778 - val_accuracy: 0.6317 - val_loss: 0.8527
Epoch 3/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6085 - loss: 0.8669 - val_accuracy: 0.6346 - val_loss: 0.8485
Epoch 4/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6264 - loss: 0.8273 - val_accuracy: 0.6308 - val_loss: 0.8396
Epoch 5/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6232 - loss: 0.8263 - val_accuracy: 0.6308 - val_loss: 0.8391
Epoch 6/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6247 - loss: 0.8321 - val_accuracy: 0.6269 - val_loss: 0.8450
Epoch 7/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6152 - loss: 0.8483 - val_accuracy: 0.6192 - val_loss: 0.8684
Epoch 8/50
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6154 - loss: 0.8482 -

In [ ]:
# 5. EVALUATE ON TEST DATA

print("\n" + "="*50)
print("TESTING MODEL")
print("="*50)

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")


TESTING MODEL
Test Accuracy: 0.6115
Test Loss: 0.9025


In [ ]:
# Make predictions
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.58      0.60       320
           1       0.45      0.41      0.43       210
           2       0.17      0.01      0.02       210
           3       0.65      0.93      0.77       560

    accuracy                           0.61      1300
   macro avg       0.47      0.48      0.45      1300
weighted avg       0.53      0.61      0.55      1300



In [ ]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[187  92   0  41]
 [ 76  86   1  47]
 [ 16   2   2 190]
 [ 21  10   9 520]]


In [ ]:
# 9. TEST MODEL WITH A SIMPLE SAMPLE

print("\n" + "="*50)
print("TESTING MODEL WITH CUSTOM SAMPLE")
print("="*50)

# Create prediction function
def predict_adhd(age, gender, edu_level, family_history, sleep, activity, 
                 phone, walking, organizing, focus, coffee, learning, anxiety):
    
    # Create dataframe with exact column names
    sample_df = pd.DataFrame([[
        age, gender, edu_level, family_history, sleep, activity, phone,
        walking, organizing, focus, coffee, learning, anxiety
    ]], columns=[
        'Age', 'Gender', 'Educational_Level', 'Family_History', 'Sleep_Hours',
        'Daily_Activity_Hours', 'Daily_Phone_Usage_Hours',
        'Daily_Walking_Running_Hours', 'Difficulty_Organizing_Tasks',
        'Focus_Score_Video', 'Daily_Coffee_Tea_Consumption',
        'Learning_Difficulties', 'Anxiety_Depression_Levels'
    ])
    
    # Encode categorical columns
    for col in label_encoders:
        if col in sample_df.columns:
            sample_df[col] = label_encoders[col].transform(sample_df[col].astype(str))
    
    # Scale features
    sample_scaled = scaler.transform(sample_df)
    
    # Predict
    prediction = model.predict(sample_scaled, verbose=0)
    predicted_class = np.argmax(prediction)
    confidence = prediction[0][predicted_class]
    
    # Diagnosis mapping
    diagnosis_map = {0: "No ADHD", 1: "Mild ADHD", 2: "Moderate ADHD", 3: "Severe ADHD"}
    
    return {
        'diagnosis': diagnosis_map[predicted_class],
        'class': predicted_class,
        'confidence': f"{confidence:.2%}",
        'probabilities': {
            'No ADHD': f"{prediction[0][0]:.2%}",
            'Mild ADHD': f"{prediction[0][1]:.2%}" if len(prediction[0]) > 1 else "0%",
            'Moderate ADHD': f"{prediction[0][2]:.2%}" if len(prediction[0]) > 2 else "0%",
            'Severe ADHD': f"{prediction[0][3]:.2%}" if len(prediction[0]) > 3 else "0%"
        }
    }

# Create ONE simple sample to test
print("\n📝 Creating test sample...")

test_sample = {
    "Age": 8,
    "Gender": 1,  # 1=Male, 2=Female
    "Educational_Level": "Primary",
    "Family_History": "Yes",
    "Sleep_Hours": 5,
    "Daily_Activity_Hours": 2,
    "Daily_Phone_Usage_Hours": 7,
    "Daily_Walking_Running_Hours": 0.5,
    "Difficulty_Organizing_Tasks": 1,
    "Focus_Score_Video": 2,
    "Daily_Coffee_Tea_Consumption": 1,
    "Learning_Difficulties": 1,
    "Anxiety_Depression_Levels": 3
}

print(f"Sample: {test_sample}")

# Make prediction
print("\n🔮 Prediction Result:")
result = predict_adhd(
    age=test_sample["Age"],
    gender=test_sample["Gender"],
    edu_level=test_sample["Educational_Level"],
    family_history=test_sample["Family_History"],
    sleep=test_sample["Sleep_Hours"],
    activity=test_sample["Daily_Activity_Hours"],
    phone=test_sample["Daily_Phone_Usage_Hours"],
    walking=test_sample["Daily_Walking_Running_Hours"],
    organizing=test_sample["Difficulty_Organizing_Tasks"],
    focus=test_sample["Focus_Score_Video"],
    coffee=test_sample["Daily_Coffee_Tea_Consumption"],
    learning=test_sample["Learning_Difficulties"],
    anxiety=test_sample["Anxiety_Depression_Levels"]
)

print(f"\n✅ Diagnosis: {result['diagnosis']}")
print(f"📊 Class: {result['class']}")
print(f"🎯 Confidence: {result['confidence']}")
print(f"\n📈 Detailed Probabilities:")
for diagnosis, prob in result['probabilities'].items():
    print(f"   {diagnosis}: {prob}")


TESTING MODEL WITH CUSTOM SAMPLE

📝 Creating test sample...
Sample: {'Age': 8, 'Gender': 1, 'Educational_Level': 'Primary', 'Family_History': 'Yes', 'Sleep_Hours': 5, 'Daily_Activity_Hours': 2, 'Daily_Phone_Usage_Hours': 7, 'Daily_Walking_Running_Hours': 0.5, 'Difficulty_Organizing_Tasks': 1, 'Focus_Score_Video': 2, 'Daily_Coffee_Tea_Consumption': 1, 'Learning_Difficulties': 1, 'Anxiety_Depression_Levels': 3}

🔮 Prediction Result:

✅ Diagnosis: Severe ADHD
📊 Class: 3
🎯 Confidence: 79.20%

📈 Detailed Probabilities:
   No ADHD: 0.07%
   Mild ADHD: 0.17%
   Moderate ADHD: 20.55%
   Severe ADHD: 79.20%
